# 06 - Gold Alerta de Pedidos Pendentes - Bike Store Medallion
Notebook de agregação da camada Gold focado em alertas de pedidos pendentes.

Fontes utilizadas da camada Silver:
- `bike_store.silver.orders`
- `bike_store.silver.clients`

## Transformações aplicadas
- Filtro: apenas pedidos com status 'Pending'
- Join com clientes que possuem e-mail e telefone cadastrados
- Resultado: lista de clientes a serem notificados

In [0]:
from pyspark.sql.functions import col, concat, lit, sum

df_orders  = spark.read.table("bike_store.silver.orders")
df_clients = spark.read.table("bike_store.silver.clients")

# Pedidos pendentes
df_pending = df_orders \
    .filter(col("order_status") == "Pending") \
    .select("order_id", "customer_id", "order_date", "quantity", "total_value")

# Join com clientes
df_gold_alertas = df_pending \
    .join(df_clients.select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone"
    ), on="customer_id", how="inner") \
    .withColumn("customer_name", concat(col("first_name"), lit(" "), col("last_name"))) \
    .drop("first_name", "last_name") \
    .orderBy("order_date")

In [0]:
# Agregando por quantidade de itens dos pedidos
df_gold_alertas_agg = df_gold_alertas \
    .groupBy("order_id", "customer_name", "email", "phone", "order_date") \
    .agg(
        sum("quantity").alias("total_items")
    ) \
    .orderBy("order_date")

display(df_gold_alertas_agg)

In [0]:
# Salvando dados em tabela
df_gold_alertas_agg.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bike_store.gold.pedidos_pendentes")

print("✅ bike_store.gold.alertas_pendentes salvo com sucesso!")